## Import librairies

In [ ]:
import numpy as np
import cvxpy as cp
import pandas as pd
from tqdm import tqdm
import plotly.express as px
from scipy.optimize import minimize
from python_module.pricing_model import BSMModel

pd.options.display.max_rows = 999
pd.options.display.max_columns = 999
pd.options.display.float_format = '{:,.2f}'.format

## Custom function

In [54]:
def compute_option_replication_bt(days_to_maturity, strike_pct, price_ts, option_type, sigma):
    
    # Prepare the time series
    price_ts.name = 'F'
    
    # Expand the time series
    orig = price_ts.reset_index()
    idx_col = orig.columns[0] 
    rows = []
    count = days_to_maturity

    for _, r in orig.iterrows():
        d = r.to_dict()
        d['days_to_maturity'] = count
        rows.append(d)
        if count == 0:

            # duplicate the same row but set count to days_to_maturity to start the next cycle
            dup = r.to_dict()
            dup['days_to_maturity'] = days_to_maturity
            rows.append(dup)

            # next appended original row should get days_to_maturity - 1
            count = days_to_maturity-1  
        else:
            count -= 1
    bt_df = pd.DataFrame(rows).set_index(idx_col)

    # Extra-outputs
    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'F0'] = bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'F']
    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'strike_date'] = bt_df.loc[bt_df['days_to_maturity']==days_to_maturity].index
    bt_df = bt_df.ffill()

    # Strike and maturity in years
    bt_df['K'] = bt_df['F0'] * strike_pct
    bt_df['T'] = bt_df['days_to_maturity'] / 252

    # Compute option delta and price
    rows = []
    for index, row in bt_df.iterrows():
        
        row_dict = row.to_dict()

        row_dict['date'] = index
        F = row_dict['F']
        K = row_dict['K']
        T = row_dict['T']

        pricing_results = BSMModel.compute_option_with_forward(
            F=F,
            K=K,
            T=T,
            r=0,
            sigma=sigma,
            option_type=option_type,
            compute_greeks=True
            )
        merged_dict = {**row_dict, **pricing_results}
        rows.append(merged_dict)
    bt_df = pd.DataFrame(rows)

    # Compute delta replication and option price changes
    bt_df['dP'] = bt_df['price'].diff()
    bt_df['dH'] = bt_df['F'].diff() * bt_df['delta'].shift(1)

    # Set dP and dH to zero on initial date
    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'dP'] = 0 
    bt_df.loc[bt_df['days_to_maturity']==days_to_maturity, 'dH'] = 0
    return bt_df

In [ ]:
def optimize_portfolio(prices, solver_type="SLSQP", target_return=None, risk_free_rate=0.0):
    """
    Optimizes portfolio weights to maximize the Sharpe Ratio (or minimize variance for QP).
    
    Args:
        prices (pd.DataFrame): Time series of asset prices (columns=assets, index=date).
        solver_type (str): 'SLSQP', 'CVXPY', 'ANALYTICAL', 'QP', 'MONTECARLO'.
        target_return (float): Required if solver_type='QP'. Annualized target return.
        risk_free_rate (float): Annualized risk-free rate (default 0.0).
        
    Returns:
        np.array: Optimal weights for the assets.
    """
    # --- 1. Data Prep ---
    # Calculate daily returns
    returns = prices.pct_change().dropna()
    
    # Annualized mean returns and covariance (Assuming 252 trading days)
    mu = returns.mean() * 252
    sigma = returns.cov() * 252
    n_assets = len(mu)
    
    # Helper to calculate portfolio stats for a given weight vector
    def get_stats(w):
        w = np.array(w)
        ret = np.sum(mu * w)
        vol = np.sqrt(np.dot(w.T, np.dot(sigma, w)))
        sr = (ret - risk_free_rate) / vol if vol > 0 else 0
        return ret, vol, sr

    # --- 2. Solver Implementations ---

    if solver_type == "ANALYTICAL":
        # Mathematical Solution (Unconstrained - Allows Short Selling)
        # Formula: z = Sigma^-1 * (mu - rf); w = z / sum(z)
        inv_sigma = np.linalg.inv(sigma)
        excess_mu = mu - risk_free_rate
        
        # Calculate unscaled weights (z)
        z = np.dot(inv_sigma, excess_mu)
        # Normalize so sum(weights) = 1
        w_opt = z / np.sum(z)
        return w_opt

    elif solver_type == "SLSQP":
        # Numerical Optimization (Constrained: Long-only)
        
        # Objective: Minimize Negative Sharpe Ratio
        def neg_sharpe(w):
            return -get_stats(w)[2]
        
        # Constraints: Sum of weights = 1
        constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        # Bounds: 0 <= w <= 1 (No short selling)
        bounds = tuple((0, 1) for _ in range(n_assets))
        init_guess = n_assets * [1. / n_assets]
        
        result = minimize(neg_sharpe, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
        return result.x

    elif solver_type == "CVXPY":
        # Convex Optimization (Robust Long-only)
        # Transformation: Minimize (1/2)x'Ex subject to (mu-rf)'x = 1, x >= 0. 
        # Then w = x / sum(x).
        
        x = cp.Variable(n_assets)
        
        # Objective: Minimize unnormalized variance
        objective = cp.Minimize(cp.quad_form(x, sigma))
        
        # Constraints
        constraints = [
            (mu.values - risk_free_rate) @ x == 1,  # Set excess return scale to 1
            x >= 0                                  # Long only
        ]
        
        prob = cp.Problem(objective, constraints)
        prob.solve()
        
        # Recover actual weights
        w_opt = x.value / np.sum(x.value)
        return w_opt

    elif solver_type == "QP":
        # Quadratic Programming (Target Return)
        # Minimizes Variance subject to a specific Target Return
        if target_return is None:
            raise ValueError("For solver_type='QP', you must provide a 'target_return'.")
            
        def portfolio_variance(w):
            return np.dot(w.T, np.dot(sigma, w))
            
        constraints = (
            {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},              # Sum weights = 1
            {'type': 'eq', 'fun': lambda w: np.sum(w * mu) - target_return} # Portfolio Return = Target
        )
        bounds = tuple((0, 1) for _ in range(n_assets))
        init_guess = n_assets * [1. / n_assets]
        
        result = minimize(portfolio_variance, init_guess, method='SLSQP', bounds=bounds, constraints=constraints)
        return result.x

    elif solver_type == "MONTECARLO":
        # Brute Force Simulation
        num_portfolios = 20000
        best_sr = -np.inf
        best_w = None
        
        for _ in range(num_portfolios):
            w = np.random.random(n_assets)
            w /= np.sum(w) # Normalize
            _, _, sr = get_stats(w)
            
            if sr > best_sr:
                best_sr = sr
                best_w = w
                
        return best_w

    else:
        raise ValueError(f"Unknown solver type: {solver_type}")

## Inputs

In [55]:
symbols = ['QQQ', 'SPY']
asset_vol = 0.10
vol_window = 20

## Import data

In [56]:
price_df = pd.DataFrame()
for symbol in symbols:
    price = pd.read_csv(f'data/{symbol}.csv', index_col=0, parse_dates=True)['price']
    price.name = symbol
    price_df = pd.concat([price_df, price], axis=1)

## Compute VT asset timeseries

In [57]:
df_log_returns = np.log(price_df).diff()
df_rolling_std = df_log_returns.ewm(span=vol_window, min_periods=vol_window, adjust=False).std()
df_rolling_std *= (252 ** 0.5)
df_leverage = asset_vol / df_rolling_std
df_leverage = df_leverage.dropna()
df_vt = (price_df.loc[df_leverage.index].pct_change() * df_leverage.shift(1)).fillna(0).add(1).cumprod() * 100
df_vt_change = df_vt.pct_change()
df_vt_level = df_vt_change.fillna(0).add(1).cumprod() * 100

## Compute building blocs

In [ ]:
i = 0
dict_bt_results = {}
for symbol in tqdm(symbols, desc='Symbol'):
      for option_type in tqdm(['call', 'put'], desc='Option Type'):
            for day_to_maturity in tqdm([5, 10, 15, 20], desc='Days to Maturity'):
                  for strike_delta in tqdm([0.01, 0.05, 0.1, 0.2], desc='Delta Strike'):
                        i += 1
                        option_name = f'{symbol}_{option_type}_DTM{day_to_maturity}_DS{strike_delta}'
                        delta_strike = strike_delta if option_type == 'call' else -strike_delta
                        strike_k = BSMModel.solve_delta_strike(F=100, T=day_to_maturity/252, sigma=asset_vol, r=0, option_type=option_type, target_delta=delta_strike)
                        strike_pct = strike_k / 100
                        df_bt = compute_option_replication_bt(
                              days_to_maturity=day_to_maturity, 
                              strike_pct=strike_pct, 
                              price_ts=df_vt_level[symbol].copy(), 
                              option_type=option_type, 
                              sigma=asset_vol)
                        df_bt['thetaT0'] = df_bt['strike_date'].map(df_bt.groupby('strike_date')['theta'].first())
                        df_bt['coeff'] = 100 / df_bt['thetaT0']
                        df_bt['dH_adj'] = df_bt['coeff']*df_bt['dH']
                        pnl = df_bt.set_index('date')['dH_adj']
                        pnl.name = option_name
                        dict_bt_results[option_name] = pnl.groupby(pnl.index).sum()
df_bt_results = pd.DataFrame(dict_bt_results)
df_bt_results = (df_bt_results / df_bt_results.min().min()) / 100

Symbol:   0%|          | 0/2 [00:00<?, ?it/s]










Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.35s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.33s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.36s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.40s/it]

Days to Maturity: 100%|██████████| 4/4 [00:21<00:00,  5.44s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.35s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.38s/it]











Delta Strike: 100%|██████████| 4/4 [00:06<00:00,  1.55s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.38s/it]

Symbol:  50%|█████     | 1/2 [00:44<00:44, 44.40s/it]










Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.47s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.38s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00:00,  1.43s/it]











Delta Strike: 100%|██████████| 4/4 [00:05<00

In [180]:
w_anal = optimize_portfolio(df_bt_results.loc[:'2019'].add(1).cumprod(), solver_type="ANALYTICAL")
best_bt = df_bt_results.dot(-w_anal)
best_bt_cumsum = best_bt.cumsum()
fig = px.line(best_bt_cumsum)
for year in best_bt.index.year.unique():
    fig.add_vline(x=pd.Timestamp(year=year, month=1, day=1), line_dash="dot", line_color="red")
fig.show()

In [181]:
x = best_bt.groupby(best_bt.index.year).sum()
x = x * max([-0.1/min(x), 0.1/max(x)])

In [182]:
px.bar(x)

In [183]:
pd.Series(index=df_bt_results.columns, data=w_anal)

QQQ_call_DTM10_DS0.1    1.13
SPY_call_DTM10_DS0.1   -0.13
dtype: float64